# Strain Gauge Data Calibration and Analysis
## Professional System Code

In [5]:
# --- Installation of Required Packages ---
!pip install ruptures plotly statsmodels scikit-learn scipy pandas numpy

In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.linear_model import HuberRegressor
from scipy.signal import savgol_filter
import ruptures as rpt
import warnings

warnings.filterwarnings('ignore')

In [7]:
def load_and_prepare_data(file_path: str) -> pd.DataFrame:
    """Loads CSV data and formats the Timestamp index."""
    df = pd.read_csv(file_path, low_memory=False)
    df.columns = df.columns.str.strip()
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df = df.sort_values('Timestamp').set_index('Timestamp')
    return df

def plot_raw_data(df: pd.DataFrame, freq_col: str, temp_col: str, title: str):
    """Plots raw frequency and temperature data with a secondary y-axis."""
    f_raw = pd.to_numeric(df[freq_col], errors='coerce').interpolate()
    t_raw = pd.to_numeric(df[temp_col], errors='coerce').interpolate()

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(x=df.index, y=f_raw, name=f"{freq_col} (Raw)", line=dict(color='blue')), secondary_y=False)
    fig.add_trace(go.Scatter(x=df.index, y=t_raw, name=f"{temp_col} (Raw)", line=dict(color='red', dash='dot')), secondary_y=True)
    
    fig.update_layout(title=title, xaxis_title="Timestamp", xaxis_rangeslider_visible=True, template="plotly_white")
    fig.update_yaxes(title_text=f"<b>{freq_col}</b>", secondary_y=False, color='blue')
    fig.update_yaxes(title_text=f"<b>{temp_col}</b>", secondary_y=True, color='red')
    return fig

In [8]:
def find_optimal_lag(df: pd.DataFrame, freq_col: str, temp_col: str, dt_hours: float, train_start_date: str, max_lag_hours: int = 4):
    """Finds the optimal time lag between temperature variations and frequency shifts."""
    df_stable = df.loc[train_start_date:].copy()
    period = int(24 / dt_hours)
    
    f_raw = pd.to_numeric(df_stable[freq_col], errors='coerce').interpolate().ffill().bfill()
    t_raw = pd.to_numeric(df_stable[temp_col], errors='coerce').interpolate().ffill().bfill()
    
    # Extract seasonal parts for accurate sync
    f_sea = seasonal_decompose(f_raw, model='additive', period=period).seasonal
    t_sea = seasonal_decompose(t_raw, model='additive', period=period).seasonal
    
    max_steps = int(max_lag_hours / dt_hours)
    shifts = np.arange(0, max_steps + 1)
    results = []

    for s in shifts:
        f_shifted = f_sea.shift(-s)
        valid_idx = f_shifted.notna() & t_sea.notna()
        if valid_idx.sum() > period:
            corr = np.corrcoef(t_sea[valid_idx], f_shifted[valid_idx])[0, 1]
            # Prioritize higher correlations with minimal delay penalties
            penalty = 1 - (s * dt_hours * 0.005)
            score = abs(corr) * penalty
            results.append({'lag_steps': s, 'lag_hrs': s * dt_hours, 'corr': corr, 'score': score})
            
    best_res = max(results, key=lambda x: x['score'])
    return best_res['lag_hrs'], best_res['corr']

In [9]:
def apply_thermal_model(df: pd.DataFrame, freq_col: str, temp_col: str, optimal_lag_hrs: float, dt_hours: float, train_start_date: str):
    """Calculates environmental/thermal fit using Huber Regression."""
    current_df = df.copy()
    shift_steps = int(optimal_lag_hrs / dt_hours)
    
    current_df['temp_input'] = current_df[temp_col].shift(shift_steps)
    current_df['temp_diff'] = current_df['temp_input'].diff().fillna(0)
    current_df['hour_rad'] = 2 * np.pi * (current_df.index.hour + current_df.index.minute/60) / 24
    
    current_df['s1'] = np.sin(current_df['hour_rad'])
    current_df['c1'] = np.cos(current_df['hour_rad'])
    current_df['s2'] = np.sin(2 * current_df['hour_rad'])
    current_df['c2'] = np.cos(2 * current_df['hour_rad'])
    
    current_df['t_s1'] = current_df['temp_input'] * current_df['s1']
    current_df['t_c1'] = current_df['temp_input'] * current_df['c1']
    current_df['t_s2'] = current_df['temp_input'] * current_df['s2']
    current_df['t_c2'] = current_df['temp_input'] * current_df['c2']
    
    feature_cols = ['temp_input', 'temp_diff', 's1', 'c1', 's2', 'c2', 't_s1', 't_c1', 't_s2', 't_c2']
    
    df_train = current_df.loc[train_start_date:].dropna(subset=feature_cols + [freq_col])
    df_predict = current_df.dropna(subset=feature_cols).copy()
    
    model = HuberRegressor(epsilon=1.1, max_iter=3000)
    model.fit(df_train[feature_cols], df_train[freq_col])
    
    df_predict['thermal_fit'] = model.predict(df_predict[feature_cols])
    df_predict['mechanical_final'] = df_predict[freq_col] - df_predict['thermal_fit']
    
    current_df = current_df.merge(df_predict[['thermal_fit', 'mechanical_final']], left_index=True, right_index=True, how='left')
    current_df['mechanical_super_smooth'] = savgol_filter(current_df['mechanical_final'].fillna(0), window_length=11, polyorder=1)
    
    return current_df, model

In [10]:
def detect_changepoints(df: pd.DataFrame, signal_col: str, min_size: int = 96, jump: int = 4, pen: float = 5, smooth_window: int = 49):
    """Detects major mechanical jump changes using PELT."""
    signal_raw = df[signal_col].fillna(method='ffill').fillna(0)
    signal_smooth = savgol_filter(signal_raw.values, window_length=smooth_window, polyorder=1)
    
    detector = rpt.Pelt(model="rbf", min_size=min_size, jump=jump)
    detector.fit(signal_smooth)
    breakpoints = detector.predict(pen=pen)
    
    step_signal = np.zeros(len(signal_smooth))
    prev = 0
    segment_info = []
    
    for bp in breakpoints:
        seg = signal_smooth[prev:bp]
        med = np.median(seg) if len(seg) > 0 else 0
        step_signal[prev:bp] = med
        segment_info.append({
            'start': df.index[prev],
            'end': df.index[min(bp, len(df)-1)],
            'median_val': med,
            'n_points': len(seg)
        })
        prev = bp
        
    step_series = pd.Series(step_signal, index=df.index)
    return step_series, breakpoints, pd.DataFrame(segment_info), signal_smooth

In [11]:
def plot_results(df: pd.DataFrame, freq_col: str, step_series: pd.Series, breakpoints: list, signal_smooth: np.ndarray):
    """Visualizes the results of thermal removal and step detection."""
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=df.index, y=df['mechanical_final'], name='Mechanical Final', line=dict(color='lightgray', width=1), opacity=0.5))
    fig.add_trace(go.Scatter(x=df.index, y=signal_smooth, name='Mechanical (SG smooth)', line=dict(color='royalblue', width=1)))
    fig.add_trace(go.Scatter(x=df.index, y=step_series, name='Step Change', line=dict(color='red', width=2.5, shape='hv')))

    bp_times = [df.index[min(bp-1, len(df)-1)] for bp in breakpoints[:-1]]
    bp_vals  = [step_series.iloc[min(bp-1, len(df)-1)] for bp in breakpoints[:-1]]

    fig.add_trace(go.Scatter(x=bp_times, y=bp_vals, mode='markers', name='Breakpoints', marker=dict(color='orange', size=8, symbol='diamond')))

    fig.update_layout(title='Mechanical Constraints - Step Change Detection', xaxis_title='Timestamp', yaxis_title='Frequency (Hz)', xaxis_rangeslider_visible=True, template="plotly_white")
    return fig

### Execution Pipeline Example

In [14]:
# 1. Provide Context & Load Data
file_path = r'C:\Users\ASUS\Downloads\SK5_SG_T.csv'
freq_col = 'Contrainte(14)'
temp_col = 'Temp(14)'
dt_hours = 0.25 # Data resolution in hours (e.g. 15 mins = 0.25)
train_start_date = '2025-01-18'

df = load_and_prepare_data(file_path)
plot_raw_data(df, freq_col, temp_col, "Raw Input Data Visualization").show()

In [15]:
# 2. Optimal Lag
optimal_lag, correlation = find_optimal_lag(df, freq_col, temp_col, dt_hours, train_start_date)
print(f"Optimal Lag: {optimal_lag} hours, Correlation: {correlation:.3f}")

Optimal Lag: 0.0 hours, Correlation: -0.965


In [16]:
# 3. Predict & Exclude Thermal Constraints
df_processed, model = apply_thermal_model(df, freq_col, temp_col, optimal_lag, dt_hours, train_start_date)

In [17]:
# 4. Identify True Mechanical Step Changes
step_series, breakpoints, segment_df, signal_smooth = detect_changepoints(df_processed, 'mechanical_final')
plot_results(df_processed, freq_col, step_series, breakpoints, signal_smooth).show()

In [18]:
# 5. Extract Step Logs
segment_df['delta_hz'] = segment_df['median_val'].diff()
mechanical_jumps = segment_df[abs(segment_df['delta_hz']) > 0.1]
display(mechanical_jumps)

,start,end,median_val,n_points,delta_hz
1,2024-11-28 16:30:00,2024-11-30 14:00:00,9.546202,96,9.546202
2,2024-11-30 14:00:00,2024-12-04 08:00:00,0.555221,96,-8.990981
3,2024-12-04 08:00:00,2024-12-08 18:30:00,-1.566127,204,-2.121348
4,2024-12-08 18:30:00,2024-12-12 09:00:00,-2.390665,140,-0.824538
5,2024-12-12 09:00:00,2024-12-13 09:00:00,1.416998,96,3.807663
6,2024-12-13 09:00:00,2024-12-14 18:36:00,2.210967,340,0.793969
7,2024-12-14 18:36:00,2024-12-14 21:48:00,1.480423,96,-0.730544
8,2024-12-14 21:48:00,2024-12-15 04:36:00,0.825547,204,-0.654876
9,2024-12-15 04:36:00,2024-12-15 14:58:00,1.307234,292,0.481688
10,2024-12-15 14:58:00,2024-12-15 18:20:00,0.940930,96,-0.366304
